# Topic: Authentication Systems - Multi-Factor Authentication (MFA), TOTP algorithm math (RFC 6238), and dynamic truncation
 
## 1. TIME-BASED ONE-TIME PASSWORDS (TOTP)
- **TOTP (RFC 6238)**: A standardized algorithm used to generate 6-digit tokens that rotate every 30 seconds.
- **Defensive Role**: Adds a second authentication factor (something you *have* - the token generator seed) to protect against credential stuffing or stolen passwords.
 
## 2. TOTP ALGORITHM STEP-BY-STEP
- **Step 1: Compute Time Step ($T$)**:
  - Divide the current Unix epoch timestamp by the time window interval ($X = 30$ seconds):
    $$T = \lfloor \frac{\text{Current Unix Time}}{30} \rfloor$$
  - Format $T$ as an 8-byte big-endian binary integer.
- **Step 2: Generate HMAC Hash**:
  - Compute the HMAC-SHA1 hash using the shared secret key ($K$) and the time step binary ($T$):
    $$\text{HS} = \text{HMAC-SHA-1}(K, T)$$
  - This produces a 20-byte binary digest.
- **Step 3: Dynamic Truncation**:
  - Extract the last 4 bits of the 20-byte digest. Use this value (offset $O$ between 0 and 15) to locate a 4-byte slice:
    $$\text{Slice} = \text{HS}[O : O+4]$$
  - Convert these 4 bytes into a 32-bit unsigned integer (ignoring the most significant sign bit).
- **Step 4: Generate 6-Digit Token**:
  - Compute modulo $10^6$ ($1,000,000$) to yield the 6-digit code. Pad with leading zeros if necessary.
 
---


In [ ]:
import hmac
import hashlib
import time
import struct
import base64

class TOTPGenerator:
    """Implements RFC 6238 TOTP validation from scratch."""
    def __init__(self, base32_secret):
        # Base32 secrets are standard in Google Authenticator setups
        # Decode secret to get raw bytes K
        # Add padding = if length is not multiple of 8
        base32_secret = base32_secret.upper()
        padding = '=' * (8 - (len(base32_secret) % 8))
        self.secret_key = base64.b32decode(base32_secret + padding)

    def get_totp_token(self, timestamp=None, interval=30):
        """Generates the 6-digit TOTP code for a given timestamp."""
        if timestamp is None:
            timestamp = time.time()
            
        # Step 1: Calculate time step T (as 8-byte big-endian int)
        time_step = int(timestamp // interval)
        time_bytes = struct.pack(">Q", time_step)  # Pack as unsigned 64-bit int
        
        # Step 2: Calculate HMAC-SHA1
        hmac_hash = hmac.new(self.secret_key, time_bytes, hashlib.sha1).digest()
        
        # Step 3: Dynamic Truncation
        # Get offset from last 4 bits of 20-byte hmac
        offset = hmac_hash[-1] & 0x0F
        
        # Extract 4-byte binary code starting at offset
        binary_code = struct.unpack(">I", hmac_hash[offset:offset+4])[0]
        # Clear most significant bit (ensure positive signed value)
        binary_code &= 0x7FFFFFFF
        
        # Step 4: Modulo 1,000,000 to get 6 digits
        token = binary_code % 1000000
        return f"{token:06d}"

    def verify_token(self, token_str, tolerance_windows=1):
        """Verifies input token, allowing tolerance windows for clock drift."""
        current_time = time.time()
        
        # Check current window, plus preceding/succeeding windows for network latency/clock drift
        for i in range(-tolerance_windows, tolerance_windows + 1):
            eval_time = current_time + (i * 30)
            if self.get_totp_token(eval_time) == token_str:
                return True
        return False



In [ ]:
# Testing the TOTP Engine
# Base32 Secret: JBSWY3DPEHPK3PXP (corresponds to ASCII bytes 'Hello!')
mfa_secret = "JBSWY3DPEHPK3PXP"
totp = TOTPGenerator(mfa_secret)

print("--- Generating TOTP Codes ---")
current_token = totp.get_totp_token()
print(f"Active TOTP Code (valid for 30s): {current_token}")

# Verify the token immediately
print(f"Verification Success? {totp.verify_token(current_token)}")  # Expected: True

# Verify against an incorrect token
print(f"Incorrect Token Verification: {totp.verify_token('999999')}")  # Expected: False
